# Bai tap: ETHUSDs (Forex)/ ETHUSDT 1m tren Forex hoac Binace
### Mot chien luoc mua: Khi gia du doan > Gia thuc te va MA 10 > MA 20
### Mot chien luoc ban: Khi gia du doan < Gia thuc te va MA 10 < MA 20

In [ ]:
# Den 21h25

# Buoc 1: Load data

In [2]:
import sys
sys.path.append('../Common')

import CommonSSIDWH

symbol = ['STB', 'ACB', 'REE', 'HAH']
from_date = '2024-11-01'
to_date = '2024-12-04' # yyyy-mm-dd
interval = '1d'

data = CommonSSIDWH.CommonSSIDWH.loaddataSSI_Ext_ListSymbol(symbol, from_date, to_date, interval)

data

Lấy dữ liệu STB từ 01/11/2024 đến 04/12/2024 thành công!
Đang xử lý mã: ACB
Lấy dữ liệu ACB từ 01/11/2024 đến 04/12/2024 thành công!
Đang xử lý mã: REE
Lấy dữ liệu REE từ 01/11/2024 đến 04/12/2024 thành công!
Đang xử lý mã: HAH
Lấy dữ liệu HAH từ 01/11/2024 đến 04/12/2024 thành công!


,Datetime,Symbol,Open,High,Low,Close,Volume
0,2024-11-01,STB,35000,35250,34950,35000,9112000
1,2024-11-04,STB,35000,35350,34650,34900,11339100
2,2024-11-05,STB,34850,35300,34850,34850,6852800
3,2024-11-06,STB,35000,35500,34800,35500,6934000
4,2024-11-07,STB,35600,35800,35400,35600,8231800
...,...,...,...,...,...,...,...
91,2024-11-28,HAH,47700,47700,47000,47650,1405400
92,2024-11-29,HAH,47600,48250,47200,48050,1955400
93,2024-12-02,HAH,48100,49200,47600,48300,2206000
94,2024-12-03,HAH,47950,49000,47600,47850,1730600


In [3]:
import pandas as pd 

data.info()
data['Open'] = pd.to_numeric(data['Open'])
data['High'] = pd.to_numeric(data['High'])
data['Low'] = pd.to_numeric(data['Low'])
data['Close'] = pd.to_numeric(data['Close'])
data['Volume'] = pd.to_numeric(data['Volume'])
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Datetime  96 non-null     datetime64[ns]
 1   Symbol    96 non-null     object        
 2   Open      96 non-null     object        
 3   High      96 non-null     object        
 4   Low       96 non-null     object        
 5   Close     96 non-null     object        
 6   Volume    96 non-null     object        
dtypes: datetime64[ns](1), object(6)
memory usage: 5.4+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Datetime  96 non-null     datetime64[ns]
 1   Symbol    96 non-null     object        
 2   Open      96 non-null     int64         
 3   High      96 non-null     int64         
 4   Low       96 non-null     int64         
 

# Buoc 2: Dua vao data:
### Tinh ra chi bao MA10, MA20
### Du doan dua va Hoi quy tuyen tinh
### Tinh Buy_Signal, Sell_Signal
### Day qua Redis

In [7]:
import talib
from sklearn.linear_model import LinearRegression 
import redis 
from datetime import datetime, timedelta
import time
import pandas as pd 
import numpy as np

for symbol in data['Symbol'].unique():

	# Lọc dữ liệu cho mã hiện tại
	symbol_data = data[data['Symbol'] == symbol].sort_values(by='Datetime').copy()  # Sao chép để tránh cảnh báo SettingWithCopy

	symbol_data['MA10'] = talib.SMA(symbol_data['Close'], timeperiod=10)
	symbol_data['MA20'] = talib.SMA(symbol_data['Close'], timeperiod=20)

	# Xu ly NA: Drop NA
	symbol_data = symbol_data.dropna()

	# Khoi tao Feature va Target
	X = symbol_data[['Open', 'High', 'Low', 'Volume', 'MA10', 'MA20']] # Feature
	y = symbol_data['Close'] # Target

	model = LinearRegression()
	model.fit(X, y)

	# Du doan
	symbol_data['Predicted_Close'] = model.predict(X)

	# Tinh toan Buy_Signal va Sell Signal
	symbol_data['Buy_Signal'] = ((symbol_data['Predicted_Close'] > symbol_data['Close']) & (symbol_data['MA10'] > symbol_data['MA20']))
	symbol_data['Sell_Signal'] = ((symbol_data['Predicted_Close'] < symbol_data['Close']) & (symbol_data['MA10'] < symbol_data['MA20']))
	symbol_data['Buy_Signal'] = True
	# Day sang Redis
	# Tạo kết nối Redis
	r = redis.Redis(host='localhost', port=6379, db=15) # DB muon chua # CK 0-5; FX 6-10; Crypto 11-15
	# Tạo hash key
	hash_key = 'OG_ML_CRTO_MA10, MA20' + '_' + symbol

	last_record = symbol_data.iloc[-1] # Lay record moi nhat

	# Chuyển đổi record cuối cùng thành từ điển và lưu vào Redis dưới dạng hash
	if(last_record['Buy_Signal'] == True or last_record['Sell_Signal'] == True):
		for field, value in last_record.to_dict().items():
			# Chuyển đổi giá trị uint64 và Timestamp thành chuỗi
			if isinstance(value, pd.Timestamp):
				value = value.isoformat()
			elif isinstance(value, (int, np.uint64)):
				value = str(value)
			r.hset(hash_key, field, value)  
			r.hset(hash_key, 'Symbol', symbol)
			r.hset(hash_key, 'Insertdate', datetime.now().isoformat())
			last_record['Insertdate'] = datetime.now().isoformat()
		print(last_record)   
	else:
		print(last_record)   
		print('Không có tín hiệu!')
	
	print('Xem Redis!')
	time.sleep(15)

C:\Users\dongn\AppData\Local\Temp\ipykernel_31664\2341514945.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  last_record['Insertdate'] = datetime.now().isoformat()
C:\Users\dongn\AppData\Local\Temp\ipykernel_31664\2341514945.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  last_record['Insertdate'] = datetime.now().isoformat()


Datetime                  2024-12-04 00:00:00
Symbol                                    STB
Open                                    32550
High                                    32600
Low                                     32150
Close                                   32400
Volume                                5780600
MA10                                  32950.0
MA20                                  33160.0
Predicted_Close                       32400.0
Buy_Signal                               True
Sell_Signal                             False
Insertdate         2024-12-04T22:19:21.297395
Name: 23, dtype: object
Xem Redis!


C:\Users\dongn\AppData\Local\Temp\ipykernel_31664\2341514945.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  last_record['Insertdate'] = datetime.now().isoformat()
C:\Users\dongn\AppData\Local\Temp\ipykernel_31664\2341514945.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  last_record['Insertdate'] = datetime.now().isoformat()


Datetime                  2024-12-04 00:00:00
Symbol                                    ACB
Open                                    24950
High                                    25050
Low                                     24850
Close                                   25050
Volume                                4616000
MA10                                  25030.0
MA20                                  24857.5
Predicted_Close                       25050.0
Buy_Signal                               True
Sell_Signal                             False
Insertdate         2024-12-04T22:19:36.321014
Name: 47, dtype: object
Xem Redis!


C:\Users\dongn\AppData\Local\Temp\ipykernel_31664\2341514945.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  last_record['Insertdate'] = datetime.now().isoformat()
C:\Users\dongn\AppData\Local\Temp\ipykernel_31664\2341514945.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  last_record['Insertdate'] = datetime.now().isoformat()


Datetime                  2024-12-04 00:00:00
Symbol                                    REE
Open                                    68000
High                                    68000
Low                                     67200
Close                                   67200
Volume                                 491300
MA10                                  66630.0
MA20                                  65595.0
Predicted_Close                       67200.0
Buy_Signal                               True
Sell_Signal                             False
Insertdate         2024-12-04T22:19:51.344044
Name: 71, dtype: object
Xem Redis!


C:\Users\dongn\AppData\Local\Temp\ipykernel_31664\2341514945.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  last_record['Insertdate'] = datetime.now().isoformat()
C:\Users\dongn\AppData\Local\Temp\ipykernel_31664\2341514945.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  last_record['Insertdate'] = datetime.now().isoformat()


Datetime                  2024-12-04 00:00:00
Symbol                                    HAH
Open                                    47700
High                                    50000
Low                                     47200
Close                                   50000
Volume                                4863400
MA10                                  47675.0
MA20                                  47027.5
Predicted_Close                       50000.0
Buy_Signal                               True
Sell_Signal                             False
Insertdate         2024-12-04T22:20:06.384385
Name: 95, dtype: object
Xem Redis!


In [5]:
data

,Datetime,Symbol,Open,High,Low,Close,Volume
0,2024-11-01,STB,35000,35250,34950,35000,9112000
1,2024-11-04,STB,35000,35350,34650,34900,11339100
2,2024-11-05,STB,34850,35300,34850,34850,6852800
3,2024-11-06,STB,35000,35500,34800,35500,6934000
4,2024-11-07,STB,35600,35800,35400,35600,8231800
...,...,...,...,...,...,...,...
91,2024-11-28,HAH,47700,47700,47000,47650,1405400
92,2024-11-29,HAH,47600,48250,47200,48050,1955400
93,2024-12-02,HAH,48100,49200,47600,48300,2206000
94,2024-12-03,HAH,47950,49000,47600,47850,1730600


In [6]:
data.to_csv('data.csv')